# Natural Experiment Statistical Analysis

Three analyses:
1. Disagreement rates, both-wrong rates, and wrong O/E ratio between two models.
2. Verification bias table per (model_family, scale).
3. Verification bias table per (model_family, scale, reasoning_level).

## 1. Load and clean data

In [ ]:
import os
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import yaml
import numpy as np
import pandas as pd
from tqdm import tqdm
from scipy.stats import fisher_exact

def find_project_root(marker=".git"):
    path = os.getcwd()
    while path != os.path.dirname(path):
        if marker in os.listdir(path):
            return path
        path = os.path.dirname(path)
    return None

root = find_project_root()
if root is not None:
    os.chdir(root)

from src.origins.natural_experiment_data import *
from src.origins.natural_experiment import *

%load_ext autoreload
%autoreload 2

In [ ]:
BASE_FOLDER = "data/natural_experiment_results"

def get_experiment_stats(folder):
    with open(f'{folder}/generation_evaluations.yaml', 'r') as f:
        gen_results_eval_year = yaml.safe_load(f)
    with open(f'{folder}/verification_evaluations.yaml', 'r') as f:
        ver_results_eval_year = yaml.safe_load(f)
    with open(f'{folder}/noise_verification_evaluations.yaml', 'r') as f:
        noise_ver_results_eval_year = yaml.safe_load(f)
    with open(f'{folder}/config.yaml', 'r') as f:
        config = yaml.safe_load(f)
    with open(f'{folder}/unique_data_ids.yaml', 'r') as f:
        unique_ids = yaml.safe_load(f)

    years = get_year_range(
        start_year=config.get('start_year'),
        end_year=config.get('end_year'),
        skip_year_frequency=config.get('skip_year_frequency')
    )
    return {
        'gen_results_eval_year': gen_results_eval_year,
        'ver_results_eval_year': ver_results_eval_year,
        'noise_ver_results_eval_year': noise_ver_results_eval_year,
        'config': config,
        'unique_ids': unique_ids,
        'years': years,
    }

def get_verification_booleans(ver_results_eval_year, noise_ver_results_eval_year, year):
    vc = ver_results_eval_year[year]['verification_correct_bools']
    vci = ver_results_eval_year[year]['verification_incorrect_bools']
    nvc = noise_ver_results_eval_year[year]['verification_correct_bools']
    nvci = noise_ver_results_eval_year[year]['verification_incorrect_bools']
    correct_bools, control_bools = [], []
    for c1, c2, n1, n2 in zip(vc, vci, nvc, nvci):
        correct_bools.append(c1 == True and c2 == False)
        control_bools.append(n1 == False and n2 == True)
    return correct_bools, control_bools

def get_latest_run(base_dir):
    path = Path(base_dir)
    subdirs = [d for d in path.iterdir() if d.is_dir()]
    if not subdirs:
        return None
    return max(subdirs, key=lambda d: d.name)

def process_folder(folder):
    try:
        fpath = os.path.join(BASE_FOLDER, folder)
        latest = get_latest_run(fpath)
        filename = os.path.basename(latest)
        exp_data = get_experiment_stats(str(latest))

        years = exp_data["years"]
        unique_ids = exp_data["unique_ids"]["unique_data_ids"]
        num_per_year = exp_data["config"]["num_data_points_per_year"]
        per_year_unique = get_datapoints_for_year(
            unique_ids,
            start_year=years[0],
            end_year=years[-1],
            skip_year_frequency=1,
            num_data_points_per_year=num_per_year,
        )

        dfs = []
        for year in years:
            correct_bools, control_bools = get_verification_booleans(
                exp_data["ver_results_eval_year"],
                exp_data["noise_ver_results_eval_year"],
                year,
            )
            df = pd.DataFrame({
                "fact_id": per_year_unique[year],
                "generation": exp_data["gen_results_eval_year"][year]["verdicts_converted"],
                "verification": correct_bools,
                "noisy_verification": control_bools,
            })
            df["model_id"] = exp_data["config"]["model_name"]
            df["fact_category"] = exp_data["config"]["dataset"]
            df["fact_year"] = year
            df["reasoning_level"] = exp_data["config"].get("model_reasoning_effort")
            df["file_name"] = filename
            dfs.append(df)
        return pd.concat(dfs, ignore_index=True)
    except Exception as e:
        print(f"Error processing folder {folder}: {e}")
        return None

def fill_model_details(x):
    mapping = {
        "gemini-3.1-pro":        ("google", "large"),
        "gemini-3-flash":        ("google", "medium"),
        "gemini-3.1-flash-lite": ("google", "small"),
        "gpt-5.4":               ("openai", "large"),
        "gpt-5.4-mini":          ("openai", "medium"),
        "gpt-5.4-nano":          ("openai", "small"),
    }
    model_id = x["model_id"]
    if model_id not in mapping:
        raise ValueError(f"Unknown model_id: {model_id}")
    family, scale = mapping[model_id]
    return pd.Series({"model_family": family, "scale": scale})

In [ ]:
folders = [
    f for f in os.listdir(BASE_FOLDER)
    if f != "example" and os.path.isdir(os.path.join(BASE_FOLDER, f))
]

data = []
with ThreadPoolExecutor(max_workers=8) as executor:
    futures = {executor.submit(process_folder, folder): folder for folder in folders}
    for future in tqdm(as_completed(futures), total=len(folders)):
        result = future.result()
        if result is not None:
            data.append(result)
data = pd.concat(data, ignore_index=True)

data[["model_family", "scale"]] = data.apply(fill_model_details, axis=1)
data.groupby(["model_id", "reasoning_level", "fact_category"])["fact_category"].count()

## 2. Disagreement, both-wrong, and wrong O/E ratio

Two-model comparison on the flagship pair (Gemini 3 Flash vs GPT-5.4, low reasoning).

In [ ]:
def get_disagreement_percentages(df, target_col):
    models = df['model_id'].unique()
    if len(models) != 2:
        raise ValueError(f"Expected exactly 2 models, got {len(models)}")
    m1, m2 = models[0], models[1]

    wide = df.pivot_table(
        index=['fact_id', 'fact_category'],
        columns='model_id',
        values=target_col,
        aggfunc='first',
    ).reset_index().dropna(subset=[m1, m2])

    # Disagreement = models differ AND one is correct (so correct_pct sums to 100%).
    wide['is_disagreement'] = (wide[m1] != wide[m2]) & ((wide[m1] == 1) | (wide[m2] == 1))

    total = len(wide)
    dis = wide[wide['is_disagreement']]
    overall_rate = (len(dis) / total) * 100 if total > 0 else 0
    print(f"Overall Disagreement Rate: {overall_rate:.2f}% ({len(dis)}/{total} facts)")

    def calc_stats(g):
        total_facts = len(g)
        dis_g = g[g['is_disagreement']]
        m1_correct = (dis_g[m1] == 1).sum()
        m2_correct = (dis_g[m2] == 1).sum()
        sum_correct = m1_correct + m2_correct
        return pd.Series({
            'total_facts': total_facts,
            'total_disagreements': len(dis_g),
            'disagreement_rate_pct': (len(dis_g) / total_facts) * 100 if total_facts > 0 else 0,
            f'{m1}_correct_pct': (m1_correct / sum_correct) * 100 if sum_correct > 0 else 0,
            f'{m2}_correct_pct': (m2_correct / sum_correct) * 100 if sum_correct > 0 else 0,
        })

    return wide.groupby('fact_category').apply(calc_stats, include_groups=False).reset_index()


def get_both_wrong_percentages(df, target_col):
    models = df['model_id'].unique()
    if len(models) != 2:
        raise ValueError(f"Expected exactly 2 models, got {len(models)}")
    m1, m2 = models[0], models[1]

    wide = df.pivot_table(
        index=['fact_id', 'fact_category'],
        columns='model_id',
        values=target_col,
        aggfunc='first',
    ).reset_index().dropna(subset=[m1, m2])

    wide['both_wrong'] = (wide[m1] == wide[m2]) & ((wide[m1] == 0) | (wide[m1] == False))

    total = len(wide)
    bw = wide[wide['both_wrong']]
    overall_rate = (len(bw) / total) * 100 if total > 0 else 0
    print(f"Overall Both-Wrong Rate: {overall_rate:.2f}% ({len(bw)}/{total} facts)")

    def calc_stats(g):
        total_facts = len(g)
        bw_count = int(g['both_wrong'].sum())
        return pd.Series({
            'total_facts': total_facts,
            'both_wrong_count': bw_count,
            'both_wrong_rate_pct': (bw_count / total_facts) * 100 if total_facts > 0 else 0,
        })

    return wide.groupby('fact_category').apply(calc_stats, include_groups=False).reset_index()


def get_wrong_oe_ratio(df, target_col):
    """Observed/Expected ratio of both models being wrong on the same fact.

    O/E == 1 under independence; > 1 means errors are correlated. Fisher's exact
    test on the 2x2 wrong/right contingency table provides a p-value.
    """
    models = df['model_id'].unique()
    if len(models) != 2:
        raise ValueError(f"Expected exactly 2 models, got {len(models)}")
    m1, m2 = models[0], models[1]

    wide = df.pivot_table(
        index=['fact_id', 'fact_category'],
        columns='model_id',
        values=target_col,
        aggfunc='first',
    ).reset_index().dropna(subset=[m1, m2])

    def calc_oe(g):
        total = len(g)
        if total == 0:
            return pd.Series()
        m1_wrong = (g[m1] == 0) | (g[m1] == False)
        m2_wrong = (g[m2] == 0) | (g[m2] == False)
        m1_rate = m1_wrong.sum() / total
        m2_rate = m2_wrong.sum() / total
        expected = m1_rate * m2_rate
        both_wrong = m1_wrong & m2_wrong
        observed = both_wrong.sum() / total
        oe = observed / expected if expected > 0 else np.nan

        o_11 = both_wrong.sum()
        o_12 = (m1_wrong & ~m2_wrong).sum()
        o_21 = (~m1_wrong & m2_wrong).sum()
        o_22 = (~m1_wrong & ~m2_wrong).sum()
        odds_ratio, p_value = fisher_exact([[o_11, o_12], [o_21, o_22]])

        return pd.Series({
            'total_facts': total,
            f'{m1}_wrong_rate': m1_rate,
            f'{m2}_wrong_rate': m2_rate,
            'expected_both_wrong_rate': expected,
            'observed_both_wrong_rate': observed,
            'O_over_E_ratio': oe,
            'p_value': p_value,
            'odds_ratio': odds_ratio,
        })

    return wide.groupby('fact_category').apply(calc_oe, include_groups=False).reset_index()

In [ ]:
df = data[data['model_id'].isin(['gemini-3-flash', 'gpt-5.4']) & (data['reasoning_level'] == 'low')].copy()

for t in ['generation', 'verification', 'noisy_verification']:
    print(f"=== Disagreement on {t} ===")
    res = get_disagreement_percentages(df, target_col=t)
    display(res)
    for col in res.columns:
        if '_pct' in col:
            print(f"{col} (mean across categories): {res[col].mean():.2f}")

In [ ]:
print("Both-wrong on verification:")
display(get_both_wrong_percentages(df, target_col='verification'))
print("Both-wrong on noisy_verification:")
display(get_both_wrong_percentages(df, target_col='noisy_verification'))

In [ ]:
print("O/E ratio on verification:")
display(get_wrong_oe_ratio(df, target_col='verification').round(5))
print("O/E ratio on noisy_verification:")
display(get_wrong_oe_ratio(df, target_col='noisy_verification').round(5))

## 3. Verification bias by (family, scale)

In [ ]:
def build_bc_bias_table(raw: pd.DataFrame, reasoning_level: str = 'low') -> pd.DataFrame:
    """Macro-averaged B, C, and B-C per (family, scale) with paired SE on B-C."""
    df = raw[raw['reasoning_level'] == reasoning_level].copy()
    df = df.dropna(subset=['verification', 'noisy_verification'])
    df['verification'] = df['verification'].astype(int)
    df['noisy_verification'] = df['noisy_verification'].astype(int)
    df['diff'] = df['verification'] - df['noisy_verification']

    per_cat = df.groupby(
        ['model_family', 'scale', 'fact_category'], as_index=False
    ).agg(
        B=('verification', 'mean'),
        C=('noisy_verification', 'mean'),
        diff_mean=('diff', 'mean'),
        diff_var=('diff', 'var'),
        n=('diff', 'size'),
    )
    per_cat['se_diff_cat'] = np.sqrt(per_cat['diff_var'] / per_cat['n'])

    out = per_cat.groupby(['model_family', 'scale'], as_index=False).agg(
        B=('B', 'mean'),
        C=('C', 'mean'),
        bc_bias=('diff_mean', 'mean'),
        _ms_var=('se_diff_cat', lambda s: (s ** 2).mean()),
        _k=('se_diff_cat', 'size'),
    )
    out['bc_bias_se'] = np.sqrt(out['_ms_var'] / out['_k'])
    out = out.drop(columns=['_ms_var', '_k'])

    out['scale'] = pd.Categorical(out['scale'], categories=['large', 'medium', 'small'], ordered=True)
    return out.sort_values(['model_family', 'scale']).reset_index(drop=True)

df_bc_bias = build_bc_bias_table(data[data['reasoning_level'] == 'low'].copy(), reasoning_level='low')
df_bc_bias

## 4. Verification bias by (family, scale, reasoning_level)

In [ ]:
def build_bc_bias_reasoning_table(raw: pd.DataFrame) -> pd.DataFrame:
    df = raw.dropna(subset=['verification', 'noisy_verification']).copy()
    df['verification'] = df['verification'].astype(int)
    df['noisy_verification'] = df['noisy_verification'].astype(int)
    df['diff'] = df['verification'] - df['noisy_verification']

    group_cols = ['model_family', 'scale', 'reasoning_level']
    per_cat = df.groupby(group_cols + ['fact_category'], as_index=False).agg(
        B=('verification', 'mean'),
        C=('noisy_verification', 'mean'),
        diff_mean=('diff', 'mean'),
        diff_var=('diff', 'var'),
        n=('diff', 'size'),
    )
    per_cat['se_diff_cat'] = np.sqrt(per_cat['diff_var'] / per_cat['n'])

    out = per_cat.groupby(group_cols, as_index=False).agg(
        B=('B', 'mean'),
        C=('C', 'mean'),
        bc_bias=('diff_mean', 'mean'),
        _ms_var=('se_diff_cat', lambda s: (s ** 2).mean()),
        _k=('se_diff_cat', 'size'),
    )
    out['bc_bias_se'] = np.sqrt(out['_ms_var'] / out['_k'])
    return out.drop(columns=['_ms_var', '_k']).sort_values(group_cols).reset_index(drop=True)

df_reasoning_bias = build_bc_bias_reasoning_table(
    data[data['reasoning_level'].isin(['low', 'medium'])].copy()
)
df_reasoning_bias